In [ ]:
#It's easy to keep track of all the graphs if I split the project up into different parts that are separated into different folders and files.

#Round 2: Only viscous damping
#The differential equation here is mx''+bx'+kx=0
#dx=v, dv=-(b/m)*v - (k/m)*x
#b^2<4ac -> b^2 < 4(3)(108) -> b < 36 (sinusoidal shape)
#Before we start varying the parameters, let's start with m=3 kg, b=12 N*s/m, k=108 N/m, and a time interval of [0,10] (seconds).

In [ ]:
%matplotlib inline
%matplotlib notebook

import numpy as np
import matplotlib.pyplot as plt

#values
N=1000
h=.01
m=3
b=12
k=108
a=np.sqrt(-b**2 + 4*m*k) #intermediate value
num_cycles=(5*a)/(6*np.pi)

t=np.zeros(N)
t[0]=0

x=np.zeros(N)
x[0]=1

v=np.zeros(N)
v[0]=1

state = np.array([x[0],v[0]])

f=np.zeros(N) #analytic
f[0]=1

f_prime=np.zeros(N) #analytic
f_prime[0]=1

K=np.zeros(N) #energies
U=np.zeros(N)
E=np.zeros(N)
e_loss=[]

max_amps=[]
min_amps=[]

In [ ]:
%matplotlib inline
%matplotlib notebook

#functions
def diff(t,state):
    [x,v]=state
    dx = v
    dv = (-b/m)*v + (-k/m)*x
    return np.array([dx,dv])

#this step actually gives the updated state from rk4
def rk4(t,state):
    k1 = diff(t,
              state)
    k2 = diff(t+h/2,
              state+(h*k1/2))
    k3 = diff(t+h/2,
              state+(h*k2/2))
    k4 = diff(t+h,
              state+(h*k3))
    return state+(h/6)*(k1+2*k2+2*k3+k4)

for n in range(N-1):
    state=rk4(t[n],state)

    t[n+1]=t[n]+h
    x[n+1]=state[0]
    v[n+1]=state[1]

    f[n+1]=np.exp(-2*t[n+1])*(np.cos(a/6 * t[n+1])+(18/a)*np.sin(a/6 * t[n+1]))
    f_prime[n+1]=-2*f[n+1]+np.exp(-2*t[n+1])*(-a/6 * np.sin(a/6 * t[n+1]) + 3*np.cos(a/6 * t[n+1]))

    K[n]=(1/2)*m*(v[n]**2)
    U[n]=(1/2)*k*(x[n]**2)
    E[n]=K[n]+U[n]

#time partitions: 0->110 111->221 222->332 333->443 444->554 555->665 666->776 777->887 888->998 (omitting t[-1] shouldn't produce a huge change)

for n in range(int(np.round(num_cycles,decimals=0))):
    e_loss.append(E[0+111*n]-E[110+111*n])
    max_amps.append(max(x[0+111*n:110+111*n]))

In [ ]:
%matplotlib inline
%matplotlib notebook

#plotting RK4 solutions
plt.figure()
plt.plot(t,x,linewidth=4)
plt.title("Position vs. Time (RK4)", fontsize=24)
plt.xlabel("Time",fontsize=14)
plt.ylabel("Position",fontsize=14)
plt.xlim()
plt.ylim()
plt.axhline(y=0, color='k')
plt.axvline(x=0, color='k')
plt.savefig("osim_p1r2_pvtr.png")
plt.show()

plt.figure()
plt.plot(t,v,linewidth=4)
plt.title("Velocity vs. Time (RK4)", fontsize=24)
plt.xlabel("Time",fontsize=14)
plt.ylabel("Velocity",fontsize=14)
plt.xlim()
plt.ylim()
plt.axhline(y=0, color='k')
plt.axvline(x=0, color='k')
plt.savefig("osim_p1r2_vvtr.png")
plt.show()

plt.figure()
plt.plot(t,K,linewidth=4)
plt.title("Kinetic Energy vs. Time (RK4)", fontsize=24)
plt.xlabel("Time",fontsize=14)
plt.ylabel("Kinetic Energy",fontsize=14)
plt.xlim()
plt.ylim()
plt.axhline(y=0, color='k')
plt.axvline(x=0, color='k')
plt.savefig("osim_p1r2_kvtr.png")
plt.show()

plt.figure()
plt.plot(t,U,linewidth=4)
plt.title("Elastic Potential Energy vs. Time (RK4)", fontsize=24)
plt.xlabel("Time",fontsize=14)
plt.ylabel("Elastic Potential Energy",fontsize=14)
plt.xlim()
plt.ylim()
plt.axhline(y=0, color='k')
plt.axvline(x=0, color='k')
plt.savefig("osim_p1r2_uvtr.png")
plt.show()

plt.figure()
plt.plot(t,E,linewidth=4)
plt.title("Total Mechanical Energy vs. Time (RK4)", fontsize=24)
plt.xlabel("Time",fontsize=14)
plt.ylabel("Total Mechanical Energy",fontsize=14)
plt.xlim()
plt.ylim()
plt.axhline(y=0, color='k')
plt.axvline(x=0, color='k')
plt.savefig("osim_p1r2_evtr.png")
plt.show()

plt.figure()
for n in range(len(e_loss)):
    plt.scatter(n+1,e_loss[n],linewidth=4,color="red")
plt.title("Energy Loss vs. Time", fontsize=24)
plt.xlabel("Cycle #",fontsize=14)
plt.ylabel("Energy Loss",fontsize=14)
plt.xlim()
plt.ylim()
plt.axhline(y=0, color='k')
plt.axvline(x=0, color='k')
plt.savefig("osim_p1r2_elvcr.png")
plt.show()

plt.figure()
for n in range(len(e_loss)):
    plt.scatter(n+1,max_amps[n],linewidth=4,color="red")
plt.title("Max Amps/Decay Envelope vs. Time", fontsize=24)
plt.xlabel("Cycle #",fontsize=14)
plt.ylabel("Max Amp in Cycle",fontsize=14)
plt.xlim()
plt.ylim()
plt.axhline(y=0, color='k')
plt.axvline(x=0, color='k')
plt.savefig("osim_p1r2_mavcr.png")
plt.show()

#there is significant loss in the total energy of the system (this makes sense)
#the loss per cycle is very high initially, but goes down as the block begins to start oscillating closer to its equilibrium point
#the max amps graph gives us an exponential decay envelope, which is expected for viscous damping

In [ ]:
#%matplotlib inline
%matplotlib notebook

#plotting analytical solutions
plt.figure()
plt.plot(t,f,linewidth=4)
plt.title("Position vs. Time (Analytical)", fontsize=24)
plt.xlabel("Time",fontsize=14)
plt.ylabel("Position",fontsize=14)
plt.xlim()
plt.ylim()
plt.axhline(y=0, color='k')
plt.axvline(x=0, color='k')
plt.savefig("osim_p1r2_pvta.png")
plt.show()

plt.figure()
plt.plot(t,f_prime,linewidth=4)
plt.title("Velocity vs. Time (Analytical)", fontsize=24)
plt.xlabel("Time",fontsize=14)
plt.ylabel("Velocity",fontsize=14)
plt.xlim()
plt.ylim()
plt.axhline(y=0, color='k')
plt.axvline(x=0, color='k')
plt.savefig("osim_p1r2_vvta.png")
plt.show()

#good to see that the position and velocity graphs are the same between RK4 and analytical